# 07 · Deposition and optical depth

This notebook goes from the optical-depth integral along a path to the sum over pieces that SkewerJAX evaluates, and
from the gas cells to $\tau$. In order:

1. the derivation from the integral to the sum over pieces;
2. the deposition, step by step: selection, deposition into pieces, the sum over the pieces, and periodic boundaries;
3. closed-form checks;
4. the line-centred frames, the global velocity axis and the observed wavelength;
5. an application to simulation outputs.

The profile that every piece adds, that of a gas element, is derived in notebook 06. Runtime: ~1 min.

**Requires:** lines (01), line profiles (06), skewers (03, $a(x)$ from 02) and gas cells (04 or 05); the last section
uses a simulation snapshot (05).
**Independent of:** the summary statistics (08).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)

In [ ]:
plot_dag(('lines', 'cosmology', 'skewers', 'samplers', 'simulation', 'fields', 'profiles', 'deposition', 'tau',)); plt.show()

In [ ]:
from skewerjax.core import (LineBundle, LINES, line_wavelength_grid_A, compute_optical_depth, resolution_report, SkewerBundle, Cells,
                            cells_from_values, plane_points, analytic as an, phys)
lines = LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"], ref_line_id="HI-LyA")
print(f"{len(LINES)} transitions in lines_table.dat")
for l, dv in zip(lines.lines, np.asarray(lines.delta_v)):
    print(f"  {l.id:11s} lambda0 = {l.lambda0_A:9.4f} A  f = {l.f_osc:.4f}  A21 = {l.A21_Hz:.3e} Hz  m = {l.mass_amu:7.3f} amu  delta_v = {dv:9.1f} km/s")
from skewerjax.core import Cosmology

## 1 · From the integral to the sum over pieces

**The integral.** Along a straight comoving path, a photon is absorbed by transition $j$ of species $i$ with optical
depth (review notes, Eq. `tau_straight_los`, with $dl_{\rm phys} = a\, dx$)

$$
\tau_j(v_\dagger) = \mathcal{C}_j \int \frac{n_{\rm com}(x)}{a(x)^3}\, \phi_j\big(\nu_j(v_\dagger, x)\big)\, a(x)\, dx
= \mathcal{C}_j \int \frac{n_{\rm com}(x)}{a(x)^2}\, \phi_j\big(\nu_j(v_\dagger, x)\big)\, dx,
\qquad \mathcal{C}_j = \frac{e^2 f_j}{4\varepsilon_0 m_e c} .
$$

It is a line integral along the path of the light. At every $x$ the gas adds its line profile $\phi_j$: the Voigt
function at the detuning $\nu_j$ of the photon from the line centre $v_c(x)$ of the gas, with the widths $\sigma$ and
$\gamma$ of its temperature and transition (notebook 06, §2).

**Step 1: cut the path into pieces.** The integral is the sum of the integrals over the parts of any partition of the
path. SkewerJAX uses **pieces** $p$, of length $\ell_p$: fragments of the path within which the gas properties are
constant. Every source gives the gas as Voronoi cells (`core.Cells`: the toy models of 04, the snapshots of 05), uniform
inside each cell, so the pieces are the exact intersections of the path with the cells (§2). By default they are also
cut at the ends of the dual segments of the sampling points, one piece per cell and dual segment; the cell crossings
(`deposition="voronoi_crossings"`) keep one interval per crossing of a cell, split into equal sub-pieces.

**Step 2: what is constant within a piece, and what is not.** Constant, exactly, because a piece lies in one cell:
$n_{\rm com}$, $T$ (and so $\sigma$ and $\gamma$) and the velocity vector; the path is straight within a piece, so
$v_\parallel = \mathbf{v}\cdot\hat t$ is constant too. Not constant: $a(x)$, so $a^{-2}$ and the Hubble part of the line
centre, $v_H(x) = c\ln(a_\dagger/a(x))$, change across every piece. The line centre moves across a piece by

$$
W_p = \left\lvert\frac{dv_c}{dx}\right\rvert \ell_p = \left\lvert\frac{dv_H}{dx}\right\rvert \ell_p ,
$$

the Hubble rate times the length of the piece.

**Step 3: one profile per piece, at its centroid.** The scale factor $a$, $a^{-2}$ and $v_c$ are evaluated at the
centroid $s_p$ of every piece ($a$ interpolated in $\ln a$ between the bundle's points, where $a(x)$ was solved), with
the values of its cell. This is the midpoint rule on each piece:

$$
\tau_j(v_\dagger) \simeq \mathcal{C}_j \sum_p \frac{n_{{\rm com},p}\, \ell_p}{a_p^2}\, \phi_{j,p}(v_\dagger),
\qquad \phi_{j,p}(v_\dagger) = V\big(\nu_j(v_\dagger - v_{c,p});\, \sigma_p, \gamma_p\big),
$$

with the widths $\sigma_p$, $\gamma_p$ of the piece's cell and the exact detuning of notebook 06, §2. This is
`core.optical_depth_kernel`, mapped over skewers by `optical_depth_from_pieces`, behind
`compute_optical_depth(source, v, lines, ...)`, whose source is a `SkewerBundle` with its `Cells`, or `Pieces`.

**Error, and when the sum is exact.** Both follow from the gas being piecewise constant:

- The midpoint rule replaces the profile smeared over $W_p$ by the profile at its centre. For a Gaussian core this
  changes the peak by a relative $W_p^2/(12 b^2)$: the sum converges to the integral at second order in the piece
  length, i.e. in the spacing of the sampling points.
- $\phi$ has unit area in $\nu$, so $\int \tau\, d\nu = \mathcal{C}_j \sum_p n_{{\rm com},p}\, \ell_p\, a_p^{-2}$ for any
  partition. The column $\sum_p n_p \ell_p$ is exact, for any spacing, because $n$ is exactly constant in a cell and the
  lengths are exact.
- The sum equals the integral exactly when $a$ and $v_c$ are constant within every piece; only the Hubble flow varies
  inside a cell. It converges to the integral as $\max_p W_p/b_p \to 0$.
- The resolution check of `compute_optical_depth` (`resolution_report`) monitors exactly this ratio, with the width of
  every piece (`Pieces.width_kms`). With the simulation default of 0.25 km/s spacing, $W_p \le 0.25$ km/s, far below
  any thermal width.

§2 builds this sum step by step, on toy cells, and adds the one option that changes where the pieces absorb: a
periodic spectrum.

## 2 · Deposition and periodic boundaries, step by step

Three different grids meet in a simulated spectrum, and the deposition is what connects them:

- **the cells:** one value per Voronoi cell, the region closer to its mesh-generating point than to any other;
- **the sampling points of the `SkewerBundle`:** point $i$ stands for its *dual segment*, between the midpoints with its
  neighbours, of length $w_i$ (03);
- **the spectral grid** of $v$ on which $\tau$ is evaluated, chosen freely.

What connects them are the **pieces**: the fragments of the path inside one cell, within which the gas is constant.
Every source of gas reaches $\tau$ as cells (`core.Cells`: the toy models of 04, the snapshots of 05), and from there
by one route. The steps below run the package's own functions on toy cells in a plane, one step per cell; the cells
of a toy model or of a snapshot go through the same functions (`compute_optical_depth(bundle, v, lines, cells)` and
`SimulationSkewers` chain them). The last step makes the spectrum periodic, an option of
$\tau$.

**The cells.** Mesh-generating points in the plane $z = L/2$ of a periodic box, so the cells are the 2D Voronoi cells of
that plane, with a filament of neutral hydrogen along $x$, temperatures of $1$ to $3\times10^4$ K and random peculiar
velocities. `Cells` holds them, as `grf_cells` (04) or `snapshot_cells` (05) would.

In [ ]:
from skewerjax.core import (select_cells, voronoi_segments, cell_crossings, absorbers_from_components, bundle_frame, absorber_pieces,
                            cell_pieces)
rng = np.random.default_rng(3)
L_toy, n_toy, a_toy = 6.0, 120, 0.25                                       # periodic box [cMpc], cells, scale factor of the data
xy = rng.uniform(0.0, L_toy, (n_toy, 2))
gen = np.column_stack([xy, np.full(n_toy, L_toy / 2)])                     # every mesh-generating point on the plane z = L/2
vel = np.column_stack([rng.normal(0.0, 60.0, (n_toy, 2)), np.zeros(n_toy)])  # peculiar velocities in the plane [km/s]
n_HI = 3e-11 * np.exp(-((xy[:, 1] - L_toy / 2) / 0.8) ** 2) * rng.uniform(0.5, 1.5, n_toy) * a_toy ** 3   # a filament along x [comoving cm^-3]
T_toy = rng.uniform(1e4, 3e4, n_toy)                                       # [K]
cells = cells_from_values(gen, L_toy, {"HI": n_HI}, temperature=T_toy, velocity=vel, volumes=np.full(n_toy, (L_toy ** 2 / n_toy) ** 1.5),
                          ids=np.arange(n_toy))
print(f"{len(cells)} toy cells in a {L_toy:g} cMpc box; typical size {cells.size_cMpc[0]:.2f} cMpc; |v| up to {np.abs(vel).max():.0f} km/s")

**The skewer.** A `SkewerBundle` of nine points along $x$ in that plane, which solves $a(x)$ with the observer before the first point.

In [ ]:
cosmo_toy = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=0.31, Omega_k_0=0.0, Omega_lambda_0=0.69)
x_sk = np.linspace(0.3, L_toy - 0.3, 9)                                    # 9 sampling points along x, in the plane
pts_toy = np.stack([x_sk, np.full_like(x_sk, L_toy / 2 + 0.2), np.full_like(x_sk, L_toy / 2)], -1)[None]
sb_toy = SkewerBundle(spatial_points=jnp.asarray(pts_toy), a0=a_toy, x0=0.0, ICs_defined_at_index=0,
                      ICs_at_emission_or_absorption="absorption", cosmologies=cosmo_toy)   # observer before the first point
print(f"one skewer of {sb_toy.N_ppsk} points; line elements w_i = {np.round(np.asarray(sb_toy.line_elements[0]), 3)} cMpc; "
      f"a from {float(sb_toy.scale_factors[0, 0, 0]):.5f} to {float(sb_toy.scale_factors[0, 0, -1]):.5f}")

**1. Selection.** `select_cells` keeps the cells within reach of the points. `select_cells_near_paths` applies the same rule to a snapshot streamed from disk (§5).

In [ ]:
sel = select_cells(cells, sb_toy.spatial_points, mode="voronoi")             # the same rule as for a snapshot
print(f"selection: {len(sel)} of {len(cells)} cells can reach the skewer (within 2.5 cell sizes of a point)")

**2. Deposition.** `voronoi_segments` cuts every dual segment at the exact boundaries of the cells it crosses
(the lower envelope of the distances to the generators): one piece per (cell, dual segment). `cell_crossings` joins
the pieces of one cell across the midpoints, which leaves one interval per crossing of a cell.

In [ ]:
seg = voronoi_segments(sb_toy.spatial_points, sel.positions, L_toy, gas_size=sel.size_cMpc)   # pieces of every dual segment
cr = cell_crossings(seg, sb_toy.spatial_points)                                                   # the same pieces joined per cell
print(f"deposition: {int((seg.cell >= 0).sum())} pieces in {sb_toy.N_ppsk} dual segments (pieces per segment: "
      f"{seg.n_components[0].tolist()}); {int(cr.n_crossings[0])} cell crossings")
print(f"the pieces tile the path: sum of their lengths {seg.length[seg.cell >= 0].sum():.6f} = path length "
      f"{float(sb_toy.path_lengths[0, -1]):.6f} cMpc")

**3. Pieces.** `absorbers_from_components` gives every piece its centroid $s_p$, its cell's density and
temperature, and the projection of its cell's velocity on the path, $v_{\parallel,p} = \mathbf{v}\cdot\hat t$. The
frame of the bundle (`bundle_frame`) gives $a(s_p)$ along the path and the direction of the photons, and
`absorber_pieces` turns them into `Pieces`, what the sum runs over. `cell_pieces` runs the three steps for every
species of a `LineBundle` in one call.

In [ ]:
lya = LineBundle(["HI-LyA"], ref_line_id="HI-LyA")
ab = absorbers_from_components(seg, sb_toy.spatial_points, sel, "HI", float(lya.mass_amu[0]))   # s (centroid), n, l, T, v_par of every piece
frame = bundle_frame(sb_toy)                                                 # a(s) along the path, a_dagger = a(0), photon direction
pieces = absorber_pieces(ab, frame, lya.n_lines)                            # Pieces: what the tau sum runs over
same = cell_pieces(cells, lya, skewer_bundle=sb_toy)                        # the three steps in one call
print(f"cell_pieces (selection + deposition + pieces in one call): the same pieces, max |dl| = "
      f"{np.abs(np.sort(np.asarray(same.length_m)) - np.sort(np.asarray(pieces.length_m))).max() / phys.Mpc_to_m:.1e} cMpc")
live = ab.n[0] > 0
print(f"{int(live.sum())} pieces; v_par from {ab.v_los[0][live].min():+.0f} to {ab.v_los[0][live].max():+.0f} km/s; "
      f"Hubble widths W_p up to {float(pieces.width_kms.max()):.1f} km/s; velocity span of the path {float(pieces.velocity_span_kms[0]):.1f} km/s")

**4. The optical depth.** `core.compute_optical_depth` sums one profile per piece,

$$
\tau(v) = \sum_p d\tau_p(v),\qquad d\tau_p(v) = \mathcal{C}_j\, n_{{\rm com},p}\, \ell_p\, a_p^{-2}\, \phi_{j,p}(v),
$$

centred at the line centre of its centroid, $v_{c,p} \simeq c\ln(a_\dagger/a(s_p)) + v_{\parallel,p}$ (§1 derives
this sum and its accuracy). `per_piece=True` returns every $d\tau_p$.

In [ ]:
v_toy = np.linspace(-150.0, float(pieces.velocity_span_kms[0]) + 150.0, 1500)
tau_pieces = np.asarray(compute_optical_depth(pieces, v_toy, lya, per_piece=True, resolution_tolerance=None))[0, 0]  # (pieces, N_v)
tau_toy = tau_pieces.sum(0)
rep = resolution_report(pieces, lya)                                       # nine points: coarse on purpose
print(f"tau = sum over {tau_pieces.shape[0]} pieces: max {tau_toy.max():.3f}; integral of tau dv {tau_toy.sum() * (v_toy[1] - v_toy[0]):.2f} km/s")
print(f"resolution check: max W/b = {rep['max_ratio'][0]:.2f}, so {rep['column_fraction'][0]:.0%} of the column is in pieces wider than b/4 "
      "in velocity (a real skewer is sampled every 0.25 km/s)")

**5. Periodic boundaries.** The cells are periodic: the gas past a face of the box is the gas at the opposite face, and
the selection and the deposition use its nearest images. That is the topology of the gas, not an option. The spectrum
is not periodic by nature. The light that crosses the path once is absorbed by the Hubble flow from $v_H = 0$ at one end
to $v_H = P = c\ln(a_{\rm first}/a_{\rm last})$ at the other, so the two ends map to different velocities, and what falls
beyond them is not on the spectrum. A periodic spectrum is a choice, `velocity_period` of `compute_optical_depth`:

- `False` (the default): every piece absorbs where its line centre is;
- `True`: each skewer is periodic over its own velocity span $P$, set by the scale factors at its first and last points
  (`Pieces.velocity_span_kms`);
- a number: that period [km/s] for every skewer.

With a period, every detuning $v_\dagger - v_c$ is replaced by its nearest image in $[-P/2, P/2)$, with two effects:

- **(i)** a piece whose line centre, Hubble flow plus peculiar velocity, falls outside the period absorbs at the other
  end, at $v_c \mp P$;
- **(ii)** a piece whose line centre is inside the period, but whose profile reaches past an end, has those wings re-enter
  at the other end.

The exact periodic spectrum is the sum over all images, $\sum_n \tau_1(v_\dagger + nP)$; the nearest image keeps one of
them. The others matter only for damped wings, which the nearest image cuts at half a period from the absorber; for
forest absorbers the missing $\tau$ is negligible. The spectral grid is a separate choice: `SimulationSkewers.spectral_grid`
covers $[0, P)$ of a box, but implies no wrap.

In [ ]:
tau_wrap = np.asarray(compute_optical_depth(pieces, v_toy, lya, per_piece=True, velocity_period=True, resolution_tolerance=None))[0, 0]
P_toy = float(pieces.velocity_span_kms[0])                                 # the period: the velocity span of the path
from skewerjax.core.spectral import line_centre_velocity_kms
s_live, u_live = ab.s[0][live], ab.v_los[0][live]
v_c_toy = np.asarray(line_centre_velocity_kms(frame.a_dagger, frame.scale_factors(0, s_live), -frame.velocity(0, s_live, u_live)[None]))[0]
one = (v_toy >= 0) & (v_toy < P_toy)
dv_toy = v_toy[1] - v_toy[0]
print(f"P = c ln(a_first / a_last) = {P_toy:.1f} km/s; line centres from {v_c_toy.min():.0f} to {v_c_toy.max():.0f} km/s")
print(f"(i) {int(((v_c_toy < 0) | (v_c_toy >= P_toy)).sum())} pieces have their line centre outside [0, P): v_c = "
      f"{np.round(np.sort(v_c_toy[(v_c_toy < 0) | (v_c_toy >= P_toy)]), 1)} km/s")
print(f"without wrap, {1 - tau_toy[one].sum() / tau_toy.sum():.0%} of the absorption (integral of tau dv) falls outside [0, P); "
      f"periodic, one period holds it all: {tau_wrap.sum(0)[one].sum() * dv_toy:.1f} km/s (on the extended axis of (c): {tau_toy.sum() * dv_toy:.1f})")

In [ ]:
from skewerjax.utils.plotting import plot_deposition_steps
fig, axs, colors = plot_deposition_steps(cells, sel, sb_toy, seg, ab, frame, v_toy, tau_pieces, tau_pieces_periodic=tau_wrap,
                                         period=P_toy)
add_panel_letters(axs)
fig.suptitle("Selection, deposition into pieces, the optical depth and its periodic version, on toy cells: one colour per crossed cell",
             fontsize=21)
plt.show()
print(f"{len(colors)} cells crossed; colours from Tol's bright palette, cycled")

*Reading the figure.*

- **(a) Selection and deposition.** Every cell is outlined; the crossed ones are filled, each in its own colour, which
  it keeps in every panel. The arrows are their peculiar velocities, at their generating points. The skewer is drawn
  as in 03: sampling points, midpoints and dual segments. The vertical ticks are the ends of the pieces: the cell
  boundaries and the midpoints. The sampling points know nothing about the cells, so a dual segment usually holds
  several pieces.
- **(b) The pieces along the path**, on the same $x$ axis: the density of every piece, in its cell's colour. The
  density is constant within a piece, and the column $\sum_p n_p \ell_p$ is exact for any spacing of the points.
- **(c) The optical depth.** Every piece adds its own profile, in its cell's colour, and their sum is $\tau$. For each
  crossed cell, the open circle marks the Hubble velocity of its pieces, $v_H = c\ln(a_\dagger/a)$, and the arrow ends
  at its line centre, shifted by the line-of-sight projection of its peculiar velocity. A cell moving away from the
  observer ($v_\parallel > 0$) absorbs at larger $v$; one moving towards it, at smaller $v$. This is why the order of the
  absorbers in velocity need not be their order along the path.
- **(d) Periodic boundaries** (`velocity_period=True`), on the axis of (c). The unshaded band is one period, $[0, P)$,
  and the shaded bands are its copies. The cell near the start that moves towards the observer has $v_c < 0$, and
  the one near the far end that moves away has $v_c > P$: their absorption, beyond the ends in (c), re-enters at
  the other end (effect (i), dashed arrows). The wings of the cells near the ends that cross $0$ or $P$
  re-enter in the same way (effect (ii)). The period then holds all the absorption of the path, and the periodic
  $\tau$ (black) departs from the direct sum (thick, light) only near the ends.

**When to use which.**

- **No wrap** (the default) is the integral along the path as it is: right for any path that is not a single crossing
  of a periodic box (a light cone, a curved path, a segment of a longer sightline), and the cheapest. For a box it
  leaves out the absorption that falls beyond the ends: in 16 TNG50-4 skewers the mean flux is higher by about
  $7\times10^{-3}$ than with a periodic spectrum, and the P1D differs by up to 25 % at $k < 0.1$ s/km.
- **`velocity_period=True`** gives a spectrum periodic over the box, on a grid of one period. FFT-based estimators of a
  box need it (08). Its approximations: wrapped gas keeps its own scale factor (that of the face it belongs to), and
  only the nearest image of each piece is kept, which matters for damped wings in small boxes.

## 3 · Optical depth: checks against closed forms

Static uniform gas ($a = 1$, no velocities) has closed forms (review notes):

- a Lorentzian for $T = 0$;
- a Gaussian for $A_{21} = 0$, the `null...` lines of the table.

The gas is a set of cells whose generating points are the sampling points of four skewers, with a uniform density
and one temperature per skewer (a uniform gas is just cells with equal values); the lines of one species share the
cells.

In [ ]:
Lk = 1e-3
null = LineBundle(["nullHI-LyA", "nullHI-LyB", "nullSiIII1206"])
pts = plane_points(Lk, "y", Lk / 2, 4, 32)                                       # 4 skewers along x, at z = 0, Lk/3, 2Lk/3, Lk
sb = SkewerBundle(spatial_points=pts, manual_scale_factors=jnp.ones_like(pts[None, ..., 0]))
v = jnp.linspace(-3000.0, 3000.0, 1024)
T_of = lambda x: 10 ** (4.0 + 4.0 * x[:, 2] / Lk)                                # temperature rising with z: 1e4 to 1e8 K
gen = np.asarray(pts, dtype=np.float64).reshape(-1, 3)                           # generating points: the sampling points
F_th = cells_from_values(gen, None, {"HI": np.full(len(gen), 1e-7), "SiIII": np.full(len(gen), 1e-7)}, temperature=T_of,
                         volumes=np.full(len(gen), (Lk / 31) ** 3))
tau = compute_optical_depth(sb, v, null, F_th, a_dagger=1.0)
T_sk = jnp.asarray(T_of(np.asarray(pts)[:, 0]))
wl = line_wavelength_grid_A(v, null, 1.0, 1.0)
ref = jnp.stack([an.tau_static_thermal(wl[j], null.lambda0_A[j:j+1], null.mass_amu[j:j+1], null.f_osc[j:j+1], sb.a0, Lk * jnp.ones(4),
                                       1e-7 * jnp.ones(4), T_sk)[0] for j in range(3)])
F, Fa = np.exp(-np.asarray(tau)), np.exp(-np.asarray(ref))
fig, axs = plt.subplots(2, 3, figsize=(22, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]}, layout="constrained")
for j, l in enumerate(lines.lines):
    for k in range(4):
        c = list(TOL.values())[k]
        axs[0, j].plot(v, Fa[j, k], color=c, **MS["reference"])
        axs[0, j].plot(v, F[j, k], color=c, **MS["default"], label=f"T = {float(T_sk[k]):.0e} K" if j == 0 else None)
        axs[1, j].plot(v, F[j, k] - Fa[j, k], color=c)
    axs[0, j].set_title(l.name); axs[1, j].set_xlabel(r"$v_\dagger$ [km/s]")
axs[0, 0].set_ylabel("F"); axs[1, 0].set_ylabel(r"$F - F_{\rm analytic}$"); axs[0, 0].legend(fontsize=15)
add_panel_letters(axs); fig.suptitle("Static thermal lines: SkewerJAX (solid) on the analytic Gaussians (light); residuals < 10⁻⁸", fontsize=22); plt.show()

Radiation-dominated expansion and the convergence with the sampling are checked in
`scripts/validation/check_optical_depth_analytic.py`.

## 4 · Line-centred frames, the global velocity axis and the observed wavelength

These are three axes for the same spectrum; changing from one to another is a change of axis only.

- **The line-centred frame** of every line: each line $j$ is integrated on a grid of $v_\dagger$, its own frame. By notebook
  06, §2, the line centre of the gas at $x$ is the same $v_c(x)$ in every frame: it does not depend on $\lambda_j$
  (`core.spectral.line_centre_velocity_kms`).
- **The global velocity axis** of a reference line (here $\mathrm{Ly}\alpha$): the same observed wavelength sits at
  $v_{\rm Ly\alpha} = v_\dagger - \Delta v_j$ in the reference frame, with

$$
\Delta v_j = c\ln\frac{\lambda_{\rm Ly\alpha}}{\lambda_j}
$$

  (`LineBundle.delta_v`). `compute_optical_depth(..., line_frame="reference")` evaluates every line directly on that
  axis, line $j$ at $v + \Delta v_j$ in its own frame (`spectral.velocities_in_line_frames`).
- **The observed wavelength** (`spectral.line_wavelength_grid_A`):

$$
\lambda_{\rm obs} = \lambda_j\, \frac{a_{\rm obs}}{a_\dagger}\, e^{v_\dagger/c} .
$$

**The gas.** Four clumps along a 20 cMpc skewer at $z \approx 3$, with no peculiar velocities. Each clump has its own
width, peak density and temperature ($8\times10^3$ to $6\times10^4$ K), so that every absorber is easy to tell apart.
The cells are generated by the 8193 points of the skewer itself: the Voronoi cells of collinear points are slabs, which
cut the skewer into its dual segments.
The lines are $\mathrm{Ly}\alpha$ and Si III $\lambda$1206, with Si III 5 times less abundant to make it visible.

In [ ]:
from skewerjax.core import Cosmology
from skewerjax.utils.plotting import plot_frames_axes
eds = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=1.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
L20, n_pt, z_obs = 20.0, 8193, 3.0
xs = np.linspace(0.0, L20, n_pt)
pts = jnp.asarray(np.stack([xs, np.full_like(xs, 1.0), np.full_like(xs, 1.0)], -1)[None])
sbf = SkewerBundle(spatial_points=pts, cosmologies=eds, a0=1 / (1 + z_obs), x0=L20)      # observer beyond the far end
# four clumps (position [cMpc], width [cMpc], peak density [cm^-3], temperature [K]): each one different
clumps = [(3.0, 0.12, 1.2e-12, 8e3), (7.5, 0.25, 0.4e-12, 6e4), (12.0, 0.10, 2.0e-12, 1.5e4), (16.5, 0.20, 0.8e-12, 3e4)]
g = np.stack([n0 * np.exp(-0.5 * ((xs - c0) / w0) ** 2) for c0, w0, n0, _ in clumps])
n_x = g.sum(0) + 1e-14
T_x = (g * np.array([c[3] for c in clumps])[:, None]).sum(0) / np.maximum(g.sum(0), 1e-300) * (g.sum(0) > 1e-16) + 1e4 * (g.sum(0) <= 1e-16)
lya_si = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
F_cl = cells_from_values(np.asarray(pts[0]), None, {"HI": n_x, "SiIII": n_x / 5}, temperature=T_x,      # cells at the points
                         volumes=np.full(n_pt, (L20 / (n_pt - 1)) ** 3))
a_pts = np.asarray(sbf.scale_factors[0, 0])
a_dag = 1 / (1 + z_obs)
P_box = float(phys.c_kms * np.log(a_pts.max() / a_pts.min()))                          # the path in velocity
dv_px = P_box / 2048
v_box = (np.arange(2048) + 0.5) * dv_px + float(phys.c_kms * np.log(a_dag / a_pts.max()))
tau_own = compute_optical_depth(sbf, v_box, lya_si, F_cl, a_dagger=a_dag)                            # (a) each line in its own frame
dV = float(lya_si.delta_v[1])
v_glob = np.arange(v_box[0] - dV - 60.0, v_box[-1] + 60.0, dv_px)                        # (b) one axis that holds both lines
tau_glob = compute_optical_depth(sbf, v_glob, lya_si, F_cl, a_dagger=a_dag, line_frame="reference")  # every line at v + Delta v_j
v_cl = [float(phys.c_kms * np.log(a_dag / np.interp(c0, xs, a_pts))) for c0, *_ in clumps]
fig, axs = plot_frames_axes(v_box, np.exp(-np.asarray(tau_own)[:, 0]), lya_si, v_ref=v_glob, F_ref=np.exp(-np.asarray(tau_glob)[:, 0]),
                            a_dagger=a_dag, a_obs=1.0, centres=v_cl)
add_panel_letters(axs)
fig.suptitle(r"The same four clumps in each line's own frame (a), on the global $\mathrm{Ly}\alpha$ axis (b) and in observed wavelength (c)",
             fontsize=21)
plt.show()
print(f"path: {P_box:.1f} km/s of Hubble flow; Delta v(Si III) = {dV:.1f} km/s; clump centres v_c = {np.round(v_cl, 1)} km/s "
      "(the same in both own frames)")
print(f"(b) Si III on the Lya axis = Si III evaluated at v + Delta v in its own frame: max |dtau| = "
      f"{np.abs(np.asarray(tau_glob)[1, 0] - np.asarray(compute_optical_depth(sbf, v_glob + dV, LineBundle(['SiIII-1206'], ref_line_id='SiIII-1206'), F_cl.select_species('SiIII'), a_dagger=a_dag))[0, 0]).max():.1e}")

*Reading the figure.*

- (a) In its own frame each line absorbs at the clump's $v_c$ (dotted): the same gas at the same velocity, in every
  line.
  - Each line's width is the clump's extent in Hubble flow plus its thermal width. The compact clumps make narrow lines,
    and the hot, extended one a broad, shallow line.
  - Si III is only slightly narrower than $\mathrm{Ly}\alpha$: its thermal width is 5.3 times smaller (Si is 28 times
    heavier), but the Hubble extent is the same for both lines.
- (b) On the $\mathrm{Ly}\alpha$ axis, Si III sits $\Delta v_j = 2270$ km/s to the blue of $\mathrm{Ly}\alpha$: the same
  clumps, the same profiles, moved by $-\Delta v_j$. The axis covers both lines, so nothing is cut.
- (c) On the observed-wavelength axis the two lines are $9.2\ \text{Å} \times (a_{\rm obs}/a_\dagger)$ apart.

## 5 · Application: $\tau$ from simulation outputs

The same integrator on the gas of notebook 05. `SimulationSkewers.optical_depth(lines)` integrates every line in its
own frame over the velocity span of the box (1 km/s pixels by default), from the deposited pieces of every dual segment
(§2). It passes `velocity_period` through: the default is the direct, non-periodic sum (§2), and
`velocity_period=True` makes each skewer periodic over its span. `to_cells(species)` gives the selected cells as `core.Cells`,
the container of 05; the profiles of the gas below are its averages over the dual segments of the points (`point_values`).

In [ ]:
DATA = os.environ.get("SKEWERJAX_DATA", "/home/STORAGE")      # folder with TNG50-4/ (docs/installation.md)
from skewerjax.core import Cosmology
from skewerjax.simulations import ArepoHDF5Reader, SimulationSkewers, IntegrationOptions, HI
reader = ArepoHDF5Reader(f"{DATA}/TNG50-4", 25)
H = reader.header
cosmo = Cosmology(H0=100 * H.h * 1e6 * phys.Gyr_to_s / phys.Gpc_to_m, Omega_r_0=0.0, Omega_m_0=H.omega_m, Omega_k_0=0.0, Omega_lambda_0=H.omega_l)
L = H.box_cMpc
x = np.linspace(0.0, L, int(round(H.vmax_kms / 0.25)) + 1)
points = np.stack([np.stack([x, np.full_like(x, y), np.full_like(x, z)], -1) for y, z in [(10.0, 30.0), (25.0, 5.0), (40.0, 45.0)]])
bundle = SkewerBundle(spatial_points=jnp.asarray(points), a0=H.a, ICs_defined_at_index=0, ICs_at_emission_or_absorption="absorption", cosmologies=cosmo)
sk = SimulationSkewers.from_bundle(reader, bundle)
print(f"TNG50-4 snapshot 25: three skewers along x, {len(sk.cells):,} cells selected in {sk.selection_s:.1f} s")

In [ ]:
lines = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
velocities, tau = sk.optical_depth(lines)
flux = np.exp(-tau)
from skewerjax.simulations import ion_from_species
pv = {sp: sk.point_values(ion_from_species(sp)) for sp in lines.species_keys}   # the gas averaged over every dual segment
a_pts = np.asarray(bundle.quadri_vectors[0][..., 0])
v_pts = phys.c_kms * np.log(a_pts[:, :1] / a_pts)
fig, axs = plt.subplots(3, 3, figsize=(22, 12), sharex=True, sharey="row", layout="constrained")
for k in range(3):
    for j, l in enumerate(lines.lines):
        sty = MS["default"] if j == 0 else MS["matched"]
        axs[0, k].semilogy(v_pts[k], pv[l.species]["n"][k], color=EC[l.species], **sty)
        axs[2, k].plot(velocities, flux[j, k], color=EC[l.species], **sty, label=l.id)
    axs[1, k].semilogy(v_pts[k], pv["HI"]["T"][k], color=EC["temperature"])
    axs[0, k].set_title(f"skewer {k}: y = {points[k, 0, 1]:.0f}, z = {points[k, 0, 2]:.0f} cMpc", fontsize=18); axs[2, k].set_xlabel("v [km/s]")
axs[0, 0].set_ylabel(r"$n$ [cm$^{-3}$]"); axs[1, 0].set_ylabel("T [K]"); axs[2, 0].set_ylabel("F"); axs[2, 0].legend()
add_panel_letters(axs, size=16); fig.suptitle("Three skewers through TNG50-4: the gas (top) and the flux it causes (bottom)", fontsize=22); plt.show()
print(f"<F>: Lya {flux[0].mean():.3f}, Si III {flux[1].mean():.4f}")